# Notebook 1: Prompt Engineering Lab

Most people jump straight to ChatGPT and type a question. In this session we slow down and ask a different question: **is the prompt good enough to trust?**

You will write prompts, run them, compare answers, and score what comes back. Bring your `.env` file (one API key) or Ollama — either is fine.


## What you will practise

- Summarising, classifying, and reasoning with prompts
- Turning vague prompts into clear instructions
- Comparing weak vs strong prompts on the same task
- Scoring outputs with a simple rubric
- Seeing why **testing** matters more than clever wording


## Step 1: Set up your notebook

1. Make sure you ran `pip install -r requirements.txt` and copied `.env.example` to `.env`.
2. Run the next two cells.
3. Look at the provider dictionary. At least one entry should be `True` (often `ollama` if you run models locally).


In [1]:
import sys
from pathlib import Path

# Run this cell first. It finds the project folder and loads your .env file.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

False

In [3]:
from src.llm_gateway import check_available_providers, run_llm

SELECTED_PROVIDER = "auto"  # change to "ollama", "openai", etc. if you want
check_available_providers()

{'openai': False,
 'anthropic': False,
 'gemini': False,
 'mistral': False,
 'cohere': False,
 'deepseek': False,
 'groq': False,
 'openrouter': False,
 'ollama': True}

## Step 2: Optional: check Ollama

Skip this if you are using a cloud API key only.

If Ollama is installed, open a terminal and run `ollama serve`. Pull any text model you like (`ollama pull llama3.2` or use one you already have).


In [4]:
from src.llm_gateway import check_ollama_server, resolve_ollama_model

check_ollama_server()
print("Using model:", resolve_ollama_model())


Using model: llava:latest


## Step 3: Start with a weak prompt

We use a short reading about generative AI. First we ask the model in the laziest way possible — on purpose.


In [5]:
from src.utils import read_sample_text

source_text = read_sample_text("generative_ai_intro.txt")
weak_prompt = f"Summarise this text: {source_text[:700]}"

print(weak_prompt[:250], "...")


Summarise this text: Generative AI is a branch of artificial intelligence focused on creating new content such as text, images, audio, code, and video. Instead of only classifying or predicting from existing data, generative systems learn patterns fr ...


In [ ]:
from src.output_formatting import show_llm_output

weak_output = run_llm(weak_prompt, provider=SELECTED_PROVIDER)
show_llm_output(weak_output, title="Weak prompt result")


**Pause and look at the answer.**

- Did it follow a clear structure?
- Did it add facts that are not in the source?
- Would you hand this to a classmate without checking it?

Most weak prompts fail quietly. That is why we test.


## Step 4: Use a clearer prompt pattern

In class we build prompts from six parts:

`Role + Task + Context + Constraints + Output format + Quality check`

You do not need fancy language. You need **specific instructions**.


In [ ]:
show_llm_output(strong_output, title="Strong prompt result")


## Step 5: Summarisation challenge (compare five versions)

Run the versions below. After each one, give it a human score from 1–5 in the table.


In [8]:
import pandas as pd

prompt_results = []
versions = {
    "1 weak": weak_prompt,
    "2 structured": strong_prompt,
    "3 audience-aware": strong_prompt.replace("first-year", "busy postgraduate"),
    "4 faithfulness": strong_prompt + "\nIf a detail is missing, write: Not in source.",
    "5 json": strong_prompt + "\nReturn JSON with keys: summary, bullets, question.",
}

for name, p in versions.items():
    out = run_llm(p, provider=SELECTED_PROVIDER, max_tokens=700)
    prompt_results.append({
        "Version": name,
        "Human score (1-5)": None,
        "Notes": "",
        "Output preview": out[:200] + "...",
    })

pd.DataFrame(prompt_results)


,Version,Human score (1-5),Notes,Output preview
0,1 weak,None,,Generative AI refers to a subfield of artifici...
1,2 structured,None,,Summary:\nGenerative AI is a branch of artific...
2,3 audience-aware,None,,Summary:\nGenerative AI is a branch of artific...
3,4 faithfulness,None,,"Summary:\n\nGenerative AI, specifically large ..."
4,5 json,None,,"```json\n{\n ""summary"": ""Generative AI is a b..."


## Step 6: Classification (student support messages)

We will sort messages into: Academic Support, Administrative Support, Technical Support, Wellbeing Concern.

**Important:** for wellbeing messages, the model should flag escalation to a human — not give medical advice.


In [9]:
messages_df = pd.read_csv(PROJECT_ROOT / "data/sample_texts/student_feedback_examples.csv")
messages_df


,message_id,message,expected_label
0,1,I cannot access the lecture slides for module ...,Technical Support
1,2,I am feeling overwhelmed and have not been sle...,Wellbeing Concern
2,3,Could you confirm whether the assignment deadl...,Administrative Support
3,4,I do not understand the difference between pre...,Academic Support
4,5,My laptop crashes whenever I run the Jupyter k...,Technical Support
5,6,I need advice on whether I should change my di...,Academic Support
6,7,I lost my student ID card and need to know the...,Administrative Support
7,8,I have been anxious about presenting in class ...,Wellbeing Concern


In [ ]:
from src.output_formatting import show_llm_output
show_llm_output(
    run_llm(classification_prompt, provider=SELECTED_PROVIDER, max_tokens=900),
    title="Classification task",
)


## Step 7: Reasoning (without asking for hidden chain-of-thought)

Scenario: a lecturer considers giving students one extra week before a practical exam.


In [ ]:
from src.output_formatting import show_llm_output
show_llm_output(
    run_llm(reasoning_prompt, provider=SELECTED_PROVIDER, max_tokens=800),
    title="Reasoning task",
)


## Step 8: Score an answer with a rubric

Rate the structured summary (or any output you choose) from 1 to 5 on each row.


In [12]:
from src.prompt_templates import score_output_rubric, RUBRIC_CRITERIA

example_scores = {
    "Clarity": 4,
    "Relevance": 4,
    "Faithfulness": 3,
    "Completeness": 4,
    "Format control": 3,
    "Usefulness": 4,
    "Safety": 5,
}

score_output_rubric(example_scores)


,Criterion,Score (1-5)
0,Clarity,4
1,Relevance,4
2,Faithfulness,3
3,Completeness,4
4,Format control,3
5,Usefulness,4
6,Safety,5


## Step 9: LLM-as-judge (use carefully)

A model can help you critique an answer. It can also be wrong. **Always keep a human in the loop.**


In [ ]:
from src.output_formatting import show_llm_output
show_llm_output(
    run_llm(judge_prompt, provider=SELECTED_PROVIDER, max_tokens=700),
    title="LLM-as-judge critique",
)


## Prompt clinic (group activity)

Here are five bad prompts. In your group, say what is wrong and rewrite one of them using the six-part pattern.

1. Make this better.
2. Explain AI.
3. Classify these students as serious or unserious.
4. Think deeply and answer.
5. Summarise this document but make it perfect.


## Wrap-up

The skill is not writing the cleverest prompt. It is **checking whether the prompt works** before you rely on it.

In Notebook 2 you will turn a good prompt into a small app.
